In [214]:
import pandas as pd

path = "../../output/rolling_stock.xlsx" 
xls = pd.ExcelFile(path)
print("Available sheets:", xls.sheet_names)

Available sheets: ['notification', 'work_order', 'tyre_pressure', 'tyre_wear', 'airbag_pressure', 'water_ponding', 'greasing_cardan_shaft', 'cardan_shaft', 'air_standup', 'cceb', 'train_startup_test']


In [215]:
selected_sheets = ["tyre_pressure", "tyre_wear", "airbag_pressure"]

dfs = {s: xls.parse(s) for s in selected_sheets}

for name, df in dfs.items():
    print(f"{name}: shape={df.shape}")

tyre_pressure: shape=(1224, 125)
tyre_wear: shape=(1198, 206)
airbag_pressure: shape=(1206, 45)


In [216]:
bogieList = sorted([
    "203", "204", "201", "202", "206", "208", "207", "205", "209", "210", 
    "212", "213", "214", "215", "217", "218", "219", "220", "221", "222", 
    "223", "224", "225", "226", "227", "229", "230", "231", "232", "248", 
    "247", "242", "243", "241", "244", "251", "252", "253", "255", "256", 
    "257", "258", "259", "260", "261", "262", "263", "264", "265", "266", 
    "267", "268", "269", "270", "271", "272", "273", "275", "276", "277", 
    "278", "279", "280", "281", "282", "283", "284", "285", "287", "288", 
    "211", "216", "245", "249", "274", "228", "235", "246", "250", "254", 
    "286"
])

In [217]:
import re
import pandas as pd
from functools import reduce
from openpyxl.styles import PatternFill

# ---------- Helpers ----------
def normalize_bogie_sn(value):
    if pd.isna(value) or value == "":
        return None
    digits = re.findall(r"\d+", str(value))
    return digits[0] if digits else None

def flag_value(v, bogie_set):
    bogie = normalize_bogie_sn(v)
    if not bogie:
        return ""
    return "Found" if bogie in bogie_set else "Unmatched"

def get_consensus_bogie(row, bogie_num):
    """
    Consensus Rules:
    Case 1 & 2: If any system is 'Found', return that SN and 'Found'.
    Case 3: If all systems are 'Unmatched', return None and 'Unmatched'.
    """
    systems = ["tyre_pressure", "tyre_wear", "airbag_pressure"]
    found_values = []
    unmatched_values = []

    for sys in systems:
        val_col = f"{sys}.bogie{bogie_num}.bogie_sn"
        flag_col = f"{val_col}_flag"
        
        if val_col in row.index:
            val = row[val_col]
            flag = row[flag_col]
            
            if flag == "Found":
                found_values.append(val)
            elif flag == "Unmatched":
                unmatched_values.append(val)

    # Priority: Return first 'Found' value encountered
    if found_values:
        return found_values[0], "Found"
    
    # Secondary: If no 'Found', but there were 'Unmatched' values
    if unmatched_values:
        return None, "Unmatched"
    
    return None, ""

# ---------- Reference ----------
bogie_set = set(bogieList) if 'bogieList' in locals() else set()

# ---------- Sheets to combine ----------
SHEETS = {
    "tyre_pressure": dfs.get("tyre_pressure"),
    "airbag_pressure": dfs.get("airbag_pressure"),
    "tyre_wear": dfs.get("tyre_wear"),
}

# ---------- Build Combined Dataframe ----------
dfs_out = []

for prefix, df in SHEETS.items():
    if df is None or df.empty:
        continue

    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.duplicated()].copy()

    rename_map = {}
    for col in df.columns:
        car_match = re.search(r'([ei]ca)(\d+)', col, flags=re.IGNORECASE)
        bogie_match = re.search(r'bogie([1-8])', col, flags=re.IGNORECASE)
        
        if bogie_match and col.lower().endswith('.bogie_sn'):
            local_bogie = int(bogie_match.group(1))
            if car_match:
                car_num = int(car_match.group(2))
                global_bogie = (car_num - 1) * 2 + local_bogie
            else:
                global_bogie = local_bogie
            
            if 1 <= global_bogie <= 8:
                new_name = f"{prefix}.bogie{global_bogie}.bogie_sn"
                rename_map[col] = new_name

    temp = df.rename(columns=rename_map)
    active_bogie_cols = list(rename_map.values())
    keep_cols = ["workorder_id", "filename"] + [c for c in active_bogie_cols if c in temp.columns]
    temp = temp[keep_cols].copy()
    temp = temp.loc[:, ~temp.columns.duplicated()]
    
    for b_col in active_bogie_cols:
        if b_col in temp.columns:
            temp[f"{b_col}_flag"] = temp[b_col].apply(lambda v: flag_value(v, bogie_set))

    dfs_out.append(temp)

# ---------- Merge and Apply Consensus ----------
if dfs_out:
    df_final = reduce(lambda l, r: pd.merge(l, r, on=["workorder_id", "filename"], how="outer"), dfs_out)
else:
    df_final = pd.DataFrame(columns=["workorder_id", "filename"])

# Generate Master columns using Consensus Logic
master_cols_order = []
for i in range(1, 9):
    val_col = f"Master.bogie{i}.sn"
    status_col = f"Master.bogie{i}.status"
    
    # Calculate consensus for each row
    res = df_final.apply(lambda row: get_consensus_bogie(row, i), axis=1)
    df_final[val_col] = res.apply(lambda x: x[0])
    df_final[status_col] = res.apply(lambda x: x[1])
    master_cols_order.extend([val_col, status_col])

# ---------- Final Reordering with Spacer ----------
final_columns = ["workorder_id", "filename"] + master_cols_order

# Create a unique name for the spacer column
spacer_col = " " 
df_final[spacer_col] = "" # Fill with empty strings
final_columns.append(spacer_col)

systems = ["tyre_pressure", "tyre_wear", "airbag_pressure"]

for i in range(1, 9):
    for system in systems:
        b_col = f"{system}.bogie{i}.bogie_sn"
        f_col = f"{b_col}_flag"
        if b_col in df_final.columns:
            final_columns.append(b_col)
        if f_col in df_final.columns:
            final_columns.append(f_col)

# Catch-all for any other columns
for col in df_final.columns:
    if col not in final_columns:
        final_columns.append(col)

df_final = df_final[final_columns]

# ---------- Export to Excel with Styling ----------
output_file = "../../output/bogieSN_validation.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_final.to_excel(writer, index=False, sheet_name="Bogies")
    ws = writer.sheets["Bogies"]

    green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
    red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    
    # Optional: Light grey for the spacer column header to make it look like a divider
    spacer_fill = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")

    for col_idx, col_name in enumerate(df_final.columns, start=1):
        # Format the spacer column
        if col_name == " ":
            ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = 3
            ws.cell(row=1, column=col_idx).fill = spacer_fill
            continue

        # Apply color to both detailed Flags and Master Statuses
        if str(col_name).endswith("_flag") or str(col_name).endswith(".status"):
            for row in range(2, ws.max_row + 1):
                cell = ws.cell(row=row, column=col_idx)
                if cell.value == "Found":
                    cell.fill = green_fill
                elif cell.value == "Unmatched":
                    cell.fill = red_fill

In [218]:
"../../output/bogieSN_validation.xlsx"

'../../output/bogieSN_validation.xlsx'

In [219]:
df_final.columns

Index(['workorder_id', 'filename', 'Master.bogie1.sn', 'Master.bogie1.status',
       'Master.bogie2.sn', 'Master.bogie2.status', 'Master.bogie3.sn',
       'Master.bogie3.status', 'Master.bogie4.sn', 'Master.bogie4.status',
       'Master.bogie5.sn', 'Master.bogie5.status', 'Master.bogie6.sn',
       'Master.bogie6.status', 'Master.bogie7.sn', 'Master.bogie7.status',
       'Master.bogie8.sn', 'Master.bogie8.status', ' ',
       'tyre_pressure.bogie1.bogie_sn', 'tyre_pressure.bogie1.bogie_sn_flag',
       'tyre_wear.bogie1.bogie_sn', 'tyre_wear.bogie1.bogie_sn_flag',
       'airbag_pressure.bogie1.bogie_sn',
       'airbag_pressure.bogie1.bogie_sn_flag', 'tyre_pressure.bogie2.bogie_sn',
       'tyre_pressure.bogie2.bogie_sn_flag', 'tyre_wear.bogie2.bogie_sn',
       'tyre_wear.bogie2.bogie_sn_flag', 'airbag_pressure.bogie2.bogie_sn',
       'airbag_pressure.bogie2.bogie_sn_flag', 'tyre_pressure.bogie3.bogie_sn',
       'tyre_pressure.bogie3.bogie_sn_flag', 'tyre_wear.bogie3.bogie_sn',